# 01_generate_dialogues.ipynb

Generate synthetic dialogues (user profile / financial / medical) with Qwen3-30B-A3B-Instruct.

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


## Setup: dialogue generator

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset
import random
import json
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.cuda.empty_cache()
Qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B-Instruct-2507", padding_side="left")
Qwen_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-30B-A3B-Instruct-2507")
Qwen_model = Qwen_model.half().to(DEVICE)

## User-profile dialogues

In [ ]:
from itertools import product

name_list = ['James', 'Emily', 'Michael', 'Sarah', 'David', 'Jessica', 'Daniel', 'Ashley', 'Matthew', 'Amanda']
age_list = [34, 27, 45, 31, 50, 29, 38, 24, 41, 36]
city_list = ["Chicago", "Houston", "Phoenix", "Dallas", "Boston", "Denver", "Austin", "Miami", "Seattle", "Atlanta"]

stable_professions = ["Surgeon", "Pharmacist", "Archaeologist", "Dentist", "Librarian", "Pilot", "Astronaut", "Chef", "Judge", "Firefighter"]

squad_dataset = load_dataset("squad", split="train")

In [ ]:
def exact_20_words(example):
    question = example["question"]
    answer = example["answers"]["text"][0]
    total_words = len(question.split()) + len(answer.split())
    return total_words == 20

In [ ]:
squad_dataset = squad_dataset.filter(exact_20_words)

squad_dataset = squad_dataset.shuffle(seed=42).select(range(1000))

In [ ]:
def user_profile_prompt(question, answer, name, age, city):
    return f"""
Scenario: You are a chat assistant chatting with a user. Generate a multi-turn, natural conversation strictly between 150 and 200 words. Insert user's information in dialogue naturally instead of listing it as a table.

User Profile:
Name: {name}
Profession: {profession}
City: {city}

IMPORTANT:
- ONLY output the dialogue.
- DO NOT include reasoning, notes, explanations, or summaries.
- Output exactly in the following format
- DO NOT replace 'User' with name
- MUST insert name, profession and city naturally in the dialogue:

Output format:
User: ...
Assistant: ...
User: ...
Assistant: ...
"""

In [ ]:
batch_size = 32

prompts = []
metadata = []
user_profile_output = []
idx = 0
all_combinations = list(product(name_list, stable_professions, city_list))
for name, profession, city in all_combinations[:60]:
    item = squad_dataset[idx]
    question = item["question"]
    answer = item["answers"]["text"][0]
    prompts.append(user_profile_prompt(question, answer, name, profession, city))
    metadata.append({"name": name, "profession": profession, "city": city})
    idx += 1

bar = tqdm(total=len(prompts), desc="Generating dialogues")

for i in range(0, len(prompts), batch_size):
    batch_prompts = prompts[i:i+batch_size]
    inputs = Qwen_tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        outputs = Qwen_model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.9)
    dialogues = [Qwen_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

    for j, dialogue in enumerate(dialogues):
        user_profile_output.append({
            "privacy": metadata[i+j],
            "dialogue": dialogue
        })
    bar.update(len(batch_prompts))

bar.close()

In [ ]:
output_path = "data/new_user_profile_raw.json"
with open(output_path, "w") as f:
    json.dump(user_profile_output, f, indent=2)

print("Saved User_Profile raw dataset size:", len(user_profile_output))

## Financial dialogues

In [ ]:
from itertools import product

bill_list = [250, 680, 1200, 540, 980, 430, 430, 310, 1050, 720]
income_list = [78500, 62000, 95000, 85400, 150000, 70200, 98000, 55000, 110000, 88500]
investment_list = [5.2, 6.0, 4.8, 7.5, 8.0, 5.8, 6.3, 6.2, 7.1, 8.0]

In [ ]:
def bank_prompt(bill, income, investment):
    return f"""
Scenario: You are a financial assistant chatting with a user. Generate a multi-turn, natural conversation strictly between 150 and 200 words. Insert user's information in dialogue naturally instead of listing it as a table.

User Information:
Bill Amount: {bill}
Income: {income}
Investment Return Percentage: {investment}

IMPORTANT:
- ONLY output the dialogue.
- DO NOT include reasoning, notes, explanations, or summaries.
- Output exactly in the following format:

Output format:
User: ...
Assistant: ...
User: ...
Assistant: ...
"""

In [ ]:
batch_size = 32
prompts = []
metadata = []
bank_output = []

all_combinations = list(product(bill_list, income_list, investment_list))
for bill, income, investment in all_combinations[:1000]:
    prompts.append(bank_prompt(bill, income, investment))
    metadata.append({"bill": bill, "income": income, "investment": investment})

bar = tqdm(total=len(prompts), desc="Generating dialogues")

for i in range(0, len(prompts), batch_size):
    batch_prompts = prompts[i:i+batch_size]
    inputs = Qwen_tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        outputs = Qwen_model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.9)
    dialogues = [Qwen_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

    for j, dialogue in enumerate(dialogues):
        bank_output.append({
            "privacy": metadata[i+j],
            "dialogue": dialogue
        })
    bar.update(len(batch_prompts))

bar.close()
output_path = "data/bank_raw.json"
with open(output_path, "w") as f:
    json.dump(bank_output, f, indent=2)

print("\nSaved Bank raw dataset size:", len(bank_output))

## Medical dialogues

In [ ]:
symptom_list = ["cough", "headache", "chest pain", "fatigue", "fever", "dizziness", "stomachache", "back pain", "allergy", "nausea"]
solution_list = ["honey", "ibuprofen", "see doctor", "vitamin D", "rest", "hydration", "antacids", "muscle relax", "antihistamine", "ginger"]
result_list = ["lungs infection", "migraine detected", "heart rhythm irregular", "iron low", "temperature high", "blood pressure low", "stomach upset", "muscle strain detected", "allergen positive", "stomach acid high"]

In [ ]:
def ill_prompt(symptom, result, solution):
    return f"""
Scenario: You are a medical assistant chatting with a user. Generate a multi-turn, natural conversation strictly between 150 and 200 words. Insert user's information in dialogue naturally instead of listing it as a table.

User Information:
Symptom: {symptom}
Examine Result: {result}
Solution: {solution}

IMPORTANT:
- ONLY output the dialogue.
- DO NOT include reasoning, notes, explanations, or summaries.
- Output exactly in the following format:

Output format:
User: ...
Assistant: ...
User: ...
Assistant: ...
"""

In [ ]:
batch_size = 32
prompts = []
metadata = []
ill_output = []

all_combinations = list(product(symptom_list, result_list, solution_list))
for symptom, result, solution in all_combinations[:1000]:
    prompts.append(ill_prompt(symptom, result, solution))
    metadata.append({"symptom": symptom, "result": result, "solution": solution})

bar = tqdm(total=len(prompts), desc="Generating dialogues")

for i in range(0, len(prompts), batch_size):
    batch_prompts = prompts[i:i+batch_size]
    inputs = Qwen_tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        outputs = Qwen_model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.9)
    dialogues = [Qwen_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

    for j, dialogue in enumerate(dialogues):
        ill_output.append({
            "privacy": metadata[i+j],
            "dialogue": dialogue
        })
    bar.update(len(batch_prompts))

bar.close()
output_path = "data/ill_raw.json"
with open(output_path, "w") as f:
    json.dump(ill_output, f, indent=2)

print("\nSaved Ill raw dataset size:", len(ill_output))